# Demo hỏi đáp trên tài liệu bằng LLM

Notebook này làm một hệ QA mini theo pipeline:

**Tài liệu -> đọc file -> chunking -> retrieval -> LLM trả lời**


## Cách dùng nhanh

1. Đặt file tài liệu vào cùng thư mục với notebook.
2. Sửa biến `DOCUMENT_PATH`.
3. Chạy lần lượt các cell.
4. Dùng `ask('câu hỏi của bạn')`.

In [5]:
# Nếu thiếu thư viện, chạy cell này một lần rồi restart kernel nếu cần
!pip install transformers torch scikit-learn pypdf python-docx -q

In [6]:
#from google.colab import files
#uploaded = files.upload()

In [7]:
from pathlib import Path
import re
import textwrap

DOCUMENT_PATH = 'sample_document.txt'   # đổi thành file của bạn
TOP_K = 3
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

Path(DOCUMENT_PATH)

PosixPath('sample_document.txt')

In [8]:
def read_txt(path):
    return Path(path).read_text(encoding='utf-8')

def read_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(path)
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or '')
    return '\n'.join(parts)

def read_docx(path):
    import docx
    doc = docx.Document(path)
    return '\n'.join(p.text for p in doc.paragraphs)

def load_document(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == '.txt':
        return read_txt(path)
    if suffix == '.pdf':
        return read_pdf(path)
    if suffix == '.docx':
        return read_docx(path)
    raise ValueError(f'Chưa hỗ trợ định dạng: {suffix}')

raw_text = load_document("README.txt")
print(raw_text[:1000])

.. -*- mode: rst -*-

|GitHubActions| |Codecov| |CircleCI| |Nightly wheels| |Ruff| |PythonVersion| |PyPI| |DOI| |Benchmark|


.. |GitHubActions| image:: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml/badge.svg
   :target: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml?query=branch%3Amain

.. |CircleCI| image:: https://circleci.com/gh/scikit-learn/scikit-learn/tree/main.svg?style=shield
   :target: https://circleci.com/gh/scikit-learn/scikit-learn

.. |Codecov| image:: https://codecov.io/gh/scikit-learn/scikit-learn/branch/main/graph/badge.svg?token=Pk8G9gg3y9
   :target: https://codecov.io/gh/scikit-learn/scikit-learn

.. |Nightly wheels| image:: https://github.com/scikit-learn/scikit-learn/actions/workflows/wheels.yml/badge.svg?event=schedule
   :target: https://github.com/scikit-learn/scikit-learn/actions?query=workflow%3A%22Wheel+builder%22+event%3Aschedule

.. |Ruff| image:: https://img.shields.io/badge/code%20style-ruff

In [9]:
def clean_text(text):
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def chunk_text(text, chunk_size=500, overlap=80):
    text = clean_text(text)
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

chunks = chunk_text(raw_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
len(chunks), chunks[:2]

(18,
 ['.. -*- mode: rst -*- |GitHubActions| |Codecov| |CircleCI| |Nightly wheels| |Ruff| |PythonVersion| |PyPI| |DOI| |Benchmark| .. |GitHubActions| image:: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml/badge.svg :target: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml?query=branch%3Amain .. |CircleCI| image:: https://circleci.com/gh/scikit-learn/scikit-learn/tree/main.svg?style=shield :target: https://circleci.com/gh/scikit-learn/scikit-lea',
  'e/main.svg?style=shield :target: https://circleci.com/gh/scikit-learn/scikit-learn .. |Codecov| image:: https://codecov.io/gh/scikit-learn/scikit-learn/branch/main/graph/badge.svg?token=Pk8G9gg3y9 :target: https://codecov.io/gh/scikit-learn/scikit-learn .. |Nightly wheels| image:: https://github.com/scikit-learn/scikit-learn/actions/workflows/wheels.yml/badge.svg?event=schedule :target: https://github.com/scikit-learn/scikit-learn/actions?query=workflow%3A%22Wheel+builder%22+event%

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(stop_words=None)
X = vectorizer.fit_transform(chunks)

def retrieve(question, top_k=3):
    q_vec = vectorizer.transform([question])
    scores = cosine_similarity(q_vec, X).ravel()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [
        {
            'rank': i + 1,
            'chunk_id': int(idx),
            'score': float(scores[idx]),
            'text': chunks[idx],
        }
        for i, idx in enumerate(top_idx)
    ]

retrieve('How to install?', top_k=2)

[{'rank': 1,
  'chunk_id': 8,
  'score': 0.43054647222484643,
  'text': 'orking installation of NumPy and SciPy, the easiest way to install scikit-learn is using ``pip``:: pip install -U scikit-learn or ``conda``:: conda install -c conda-forge scikit-learn The documentation includes more detailed `installation instructions <https://scikit-learn.org/stable/install.html>`_. Changelog --------- See the `changelog <https://scikit-learn.org/dev/whats_new.html>`__ for a history of notable changes to scikit-learn. Development ----------- We welcome new contributors of all e'},
 {'rank': 2,
  'chunk_id': 7,
  'score': 0.12926267715169767,
  'text': 'nd classes end with ``Display``) require Matplotlib (>= |MatplotlibMinVersion|). For running the examples Matplotlib >= |MatplotlibMinVersion| is required. A few examples require scikit-image >= |Scikit-ImageMinVersion|, a few examples require pandas >= |PandasMinVersion|, some examples require seaborn >= |SeabornMinVersion| and Plotly >= |PlotlyMi

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Loaded", MODEL_NAME, "on", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded google/flan-t5-base on cpu


In [12]:
def build_prompt(question, retrieved):
    context = '\n\n'.join([f"[Chunk {r['chunk_id']}] {r['text']}" for r in retrieved])

    prompt = f"""
You are a helpful QA assistant.

Strict rules:
- Use ONLY the information from the context
- Do NOT use outside knowledge
- If unsure, say: "I cannot find this information in the document"

Context:
{context}

Question: {question}

Answer in English, concise and accurate.
""".strip()

    return prompt

def answer_with_llm(question, top_k=3, max_new_tokens=128):
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, retrieved, prompt

ans, retrieved, prompt = answer_with_llm('How to install', top_k=3)
ans

'installation instructions https://scikit-learn.org/stable/install.html>_. Changelog --------- See the changelog https://scikit-learn.org/dev/whats_new.html>__ for a history of notable changes to scikit-learn. Development -----------'

In [13]:
def ask(question, top_k=3, show_chunks=True):
    answer, retrieved, prompt = answer_with_llm(question, top_k=top_k)
    print('CÂU HỎI:')
    print(question)
    print('\nTRẢ LỜI:')
    print(answer)
    if show_chunks:
        print('\n' + '=' * 90)
        print('CÁC ĐOẠN ĐƯỢC LẤY RA:')
        for r in retrieved:
            print(f"\n[Rank {r['rank']}] chunk_id={r['chunk_id']} | score={r['score']:.4f}")
            print(textwrap.fill(r['text'], width=100))

ask('How to install?')

CÂU HỎI:
How to install?

TRẢ LỜI:
pip:: pip install -U scikit-learn or conda:: conda install -c conda-forge scikit-learn The documentation includes more detailed installation instructions https://scikit-learn.org/stable/install.html>_. Changelog --------- See the changelog https://scikit-learn.org/dev/whats_new.html>__ for a history of

CÁC ĐOẠN ĐƯỢC LẤY RA:

[Rank 1] chunk_id=8 | score=0.4305
orking installation of NumPy and SciPy, the easiest way to install scikit-learn is using ``pip``::
pip install -U scikit-learn or ``conda``:: conda install -c conda-forge scikit-learn The
documentation includes more detailed `installation instructions <https://scikit-
learn.org/stable/install.html>`_. Changelog --------- See the `changelog <https://scikit-
learn.org/dev/whats_new.html>`__ for a history of notable changes to scikit-learn. Development
----------- We welcome new contributors of all e

[Rank 2] chunk_id=7 | score=0.1293
nd classes end with ``Display``) require Matplotlib (>= |Matplo

## Thử vài câu hỏi mẫu

In [14]:
questions = [
    'What is scikit-learn?',
    'What is scikit-learn used for?',
    'What kind of problems does scikit-learn solve?'
]

for q in questions:
    ask(q, top_k=TOP_K, show_chunks=False)
    print('\n' + '-' * 90 + '\n')

CÂU HỎI:
What is scikit-learn?

TRẢ LỜI:
Python module for machine learning built on top of SciPy

------------------------------------------------------------------------------------------

CÂU HỎI:
What is scikit-learn used for?

TRẢ LỜI:
machine learning

------------------------------------------------------------------------------------------

CÂU HỎI:
What kind of problems does scikit-learn solve?

TRẢ LỜI:
- Do NOT use outside knowledge

------------------------------------------------------------------------------------------

